# 05. Post-processing

**Goal:** connected-component filtering, and why it is a dataset-specific heuristic rather than a default cleanup step.

In [ ]:
# bootstrap: make course_utils + scripts importable from any working directory,
# use the inline backend so figures render, and regenerate the phantom if missing.
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO = _find_repo_root(Path.cwd())
for _p in (str(REPO), str(REPO / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

DATA = REPO / "assets" / "data" / "Dataset999_Phantom"
PRE = REPO / "assets" / "precomputed"

if not (DATA / "imagesTr" / "PHANTOM_001_0000.nii.gz").exists():
    import generate_phantom
    generate_phantom.generate(REPO / "assets" / "data")

print("repo root:", REPO.name)


## Keep large components, drop tiny ones

nnU-Net's real post-processing is selected per dataset from cross-validation. Here we show the mechanic on the illustrative prediction, which has a couple of spurious blobs.

In [ ]:
import nibabel as nib
ct = np.asarray(nib.load(str(DATA / 'imagesTr' / 'PHANTOM_001_0000.nii.gz')).dataobj, dtype=np.float32)
gt = np.rint(np.asarray(nib.load(str(DATA / 'labelsTr' / 'PHANTOM_001.nii.gz')).dataobj)).astype(int)
from teaching_fixtures import illustrative_prediction
from scipy import ndimage
pred = illustrative_prediction(gt)
labels, n = ndimage.label(pred == 1)
sizes = ndimage.sum_labels(np.ones_like(labels), labels, index=range(1, n + 1))
print('connected components:', n, ' sizes:', sorted(map(int, sizes), reverse=True))

keep = {i + 1 for i, s in enumerate(sizes) if s >= 20}   # keep components with >= 20 voxels
clean = np.where(np.isin(labels, list(keep)), 1, 0)
print('voxels before:', int((pred == 1).sum()), ' after filter:', int((clean == 1).sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, m, t in zip(axes, [pred, clean], ['before', 'after CC filter']):
    ax.imshow(m.max(axis=2).T, origin='lower', cmap='magma')   # max projection so small blobs show
    ax.set_title(t); ax.axis('off')
fig.tight_layout()
plt.show()

## When it hurts

Aggressive component filtering can delete real multifocal lesions, metastases, or bilateral structures. Treat it as a validated, dataset-specific heuristic: measure its effect on held-out data before trusting it.

## Recap
1. Connected-component filtering removes speckle but can erase real small lesions.
2. It is chosen from cross-validation, not applied blindly.